# 06 — Kaggle Submission

Run the same original-authors `style_monet` baseline as notebook 02 on Kaggle. No training or fine-tuning.


## Want to see more?

I turned this Kaggle competition into a larger project. You can follow my work in the [GitHub repository](https://github.com/KurtMeehanPro/painter-myself).

Now back to this notebook...


## Set up Kaggle

- Create a notebook from the Painter Myself competition and import this notebook.
- Attach the **gan-getting-started** competition input and select a GPU accelerator.
- For offline use, upload `data/checkpoints/style_monet/style_monet.pth` as a Kaggle Dataset, then attach it as an input. Keep the filename `style_monet.pth`.
- Generator and helper code are included below; no project files or package installs are needed.
- Alternatively, enable Kaggle Internet and set `ALLOW_CHECKPOINT_DOWNLOAD = True` below. The default is offline.


## Notes
The [competition format](https://www.kaggle.com/competitions/gan-getting-started/overview) requires 7,000–10,000 RGB JPGs of size 256×256 and one output named `images.zip`. Kaggle requires a saved Kaggle notebook run for submission. This notebook includes the helper code and uses attached weights or an explicitly enabled download. Its working output should contain only `images.zip`. Running these cells creates the output; submission is a separate action.

## Rules and model source

- The [competition overview](https://www.kaggle.com/competitions/gan-getting-started/) requires generated images, allows internet access, and describes submission from a saved notebook run.
- The indexed [competition rules](https://www.kaggle.com/c/gan-getting-started/rules) permit external data. The full detailed rules page could not be retrieved during preparation, so a separate pretrained-model clause was not independently verified; check the linked rules before submitting.
- Public weights: [authors' downloader](https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix/blob/master/scripts/download_cyclegan_model.sh). The [model zoo](https://github.com/junyanz/CycleGAN#model-zoo) identifies `style_monet` as landscape photo → Monet style.
- All output images are generated from photos, not copied or altered Monet training paintings.


## Imports


In [ ]:
# standard library
from io import BytesIO
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile, ZIP_STORED
import functools
import hashlib
import time

# third-party
import numpy as np
from PIL import Image
import torch
from torch import nn


## Settings

- Leave the overrides as `None` for automatic input discovery.
- Set an override to a full Kaggle input path only if multiple matching inputs are attached.


In [ ]:
ALLOW_CHECKPOINT_DOWNLOAD = False
PHOTO_DIR_OVERRIDE = None
CHECKPOINT_PATH_OVERRIDE = None

input_root = Path("/kaggle/input")
output_dir = Path("/kaggle/working")
scratch_dir = Path("/kaggle/tmp/style_monet")
checkpoint_url = "https://efrosgans.eecs.berkeley.edu/cyclegan/pretrained_models/style_monet.pth"
checkpoint_sha256 = "efdad8782a185c2c423559449a2f3e33a6eeadc7d042ada98ffa848e6b055142"
output_zip_path = output_dir / "images.zip"
temporary_output_zip_path = scratch_dir / "images.partial.zip"


## Find the competition photos


In [ ]:
if not input_root.is_dir():
    raise FileNotFoundError("Run this notebook in Kaggle with the competition input attached.")
if PHOTO_DIR_OVERRIDE is not None:
    photo_dir = Path(PHOTO_DIR_OVERRIDE)
else:
    photo_dirs = sorted(path for path in input_root.rglob("photo_jpg") if path.is_dir())
    if len(photo_dirs) != 1:
        raise ValueError(f"Expected one photo_jpg directory; found {photo_dirs}. Attach the competition input or set PHOTO_DIR_OVERRIDE.")
    photo_dir = photo_dirs[0]
photo_paths = sorted(photo_dir.glob("*.jpg"))
if not 7000 <= len(photo_paths) <= 10000:
    raise ValueError(f"Expected 7,000–10,000 photos; found {len(photo_paths):,} in {photo_dir}.")
print(f"Photos: {len(photo_paths):,} — {photo_dir}")


## Find the pretrained weights

Attached weights work with Internet off. Downloads, if explicitly enabled, go to temporary storage rather than the notebook output. A SHA256 check ensures the checkpoint matches the one used in notebook 02.


In [ ]:
scratch_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
if CHECKPOINT_PATH_OVERRIDE is not None:
    checkpoint_path = Path(CHECKPOINT_PATH_OVERRIDE)
else:
    candidates = sorted(input_root.rglob("style_monet.pth"))
    if len(candidates) > 1:
        raise ValueError(f"Multiple checkpoints found: {candidates}. Set CHECKPOINT_PATH_OVERRIDE.")
    checkpoint_path = candidates[0] if candidates else scratch_dir / "style_monet.pth"

if not checkpoint_path.is_file():
    if not ALLOW_CHECKPOINT_DOWNLOAD:
        raise FileNotFoundError("Attach style_monet.pth as an input, or enable Kaggle Internet and ALLOW_CHECKPOINT_DOWNLOAD.")
    if CHECKPOINT_PATH_OVERRIDE is not None:
        raise FileNotFoundError(f"Checkpoint override not found: {checkpoint_path}")
    download_path = scratch_dir / "style_monet.download"
    try:
        urlretrieve(checkpoint_url, download_path)
    except Exception as e:
        raise RuntimeError("Checkpoint download failed. Check Kaggle Internet or attach style_monet.pth for offline use.") from e
    download_path.replace(checkpoint_path)

actual_sha256 = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
if actual_sha256 != checkpoint_sha256:
    raise ValueError(f"Checkpoint SHA256 mismatch: {actual_sha256}")
print(f"Checkpoint verified: {checkpoint_path}")


## Generator implementation

The two classes below are copied unchanged from `scripts/cyclegan_generator.py`, which reuses the original authors' code. Attribution and license are retained at the end. This is the same nine-residual-block, 64-filter, InstanceNorm generator used in notebook 02.


In [ ]:
class ResnetGenerator(nn.Module):
    """Resnet-based generator that consists of Resnet blocks between a few downsampling/upsampling operations.

    We adapt Torch code and idea from Justin Johnson's neural style transfer project(https://github.com/jcjohnson/fast-neural-style)
    """

    def __init__(self, input_nc, output_nc, ngf=64, norm_layer=nn.BatchNorm2d, use_dropout=False, n_blocks=6, padding_type="reflect"):
        """Construct a Resnet-based generator

        Parameters:
            input_nc (int)      -- the number of channels in input images
            output_nc (int)     -- the number of channels in output images
            ngf (int)           -- the number of filters in the last conv layer
            norm_layer          -- normalization layer
            use_dropout (bool)  -- if use dropout layers
            n_blocks (int)      -- the number of ResNet blocks
            padding_type (str)  -- the name of padding layer in conv layers: reflect | replicate | zero
        """
        assert n_blocks >= 0
        super(ResnetGenerator, self).__init__()
        if type(norm_layer) == functools.partial:
            use_bias = norm_layer.func == nn.InstanceNorm2d
        else:
            use_bias = norm_layer == nn.InstanceNorm2d

        model = [nn.ReflectionPad2d(3), nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0, bias=use_bias), norm_layer(ngf), nn.ReLU(True)]

        n_downsampling = 2
        for i in range(n_downsampling):  # add downsampling layers
            mult = 2**i
            model += [nn.Conv2d(ngf * mult, ngf * mult * 2, kernel_size=3, stride=2, padding=1, bias=use_bias), norm_layer(ngf * mult * 2), nn.ReLU(True)]

        mult = 2**n_downsampling
        for i in range(n_blocks):  # add ResNet blocks

            model += [ResnetBlock(ngf * mult, padding_type=padding_type, norm_layer=norm_layer, use_dropout=use_dropout, use_bias=use_bias)]

        for i in range(n_downsampling):  # add upsampling layers
            mult = 2 ** (n_downsampling - i)
            model += [nn.ConvTranspose2d(ngf * mult, int(ngf * mult / 2), kernel_size=3, stride=2, padding=1, output_padding=1, bias=use_bias), norm_layer(int(ngf * mult / 2)), nn.ReLU(True)]
        model += [nn.ReflectionPad2d(3)]
        model += [nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0)]
        model += [nn.Tanh()]

        self.model = nn.Sequential(*model)

    def forward(self, input):
        """Standard forward"""
        return self.model(input)


In [ ]:
class ResnetBlock(nn.Module):
    """Define a Resnet block"""

    def __init__(self, dim, padding_type, norm_layer, use_dropout, use_bias):
        """Initialize the Resnet block

        A resnet block is a conv block with skip connections
        We construct a conv block with build_conv_block function,
        and implement skip connections in <forward> function.
        Original Resnet paper: https://arxiv.org/pdf/1512.03385.pdf
        """
        super(ResnetBlock, self).__init__()
        self.conv_block = self.build_conv_block(dim, padding_type, norm_layer, use_dropout, use_bias)

    def build_conv_block(self, dim, padding_type, norm_layer, use_dropout, use_bias):
        """Construct a convolutional block.

        Parameters:
            dim (int)           -- the number of channels in the conv layer.
            padding_type (str)  -- the name of padding layer: reflect | replicate | zero
            norm_layer          -- normalization layer
            use_dropout (bool)  -- if use dropout layers.
            use_bias (bool)     -- if the conv layer uses bias or not

        Returns a conv block (with a conv layer, a normalization layer, and a non-linearity layer (ReLU))
        """
        conv_block = []
        p = 0
        if padding_type == "reflect":
            conv_block += [nn.ReflectionPad2d(1)]
        elif padding_type == "replicate":
            conv_block += [nn.ReplicationPad2d(1)]
        elif padding_type == "zero":
            p = 1
        else:
            raise NotImplementedError("padding [%s] is not implemented" % padding_type)

        conv_block += [nn.Conv2d(dim, dim, kernel_size=3, padding=p, bias=use_bias), norm_layer(dim), nn.ReLU(True)]
        if use_dropout:
            conv_block += [nn.Dropout(0.5)]

        p = 0
        if padding_type == "reflect":
            conv_block += [nn.ReflectionPad2d(1)]
        elif padding_type == "replicate":
            conv_block += [nn.ReplicationPad2d(1)]
        elif padding_type == "zero":
            p = 1
        else:
            raise NotImplementedError("padding [%s] is not implemented" % padding_type)
        conv_block += [nn.Conv2d(dim, dim, kernel_size=3, padding=p, bias=use_bias), norm_layer(dim)]

        return nn.Sequential(*conv_block)

    def forward(self, x):
        """Forward function (with skip connections)"""
        out = x + self.conv_block(x)  # add skip connections
        return out


## Load the generator

- Remove only obsolete InstanceNorm bookkeeping buffers; strictly load all learned parameters.
- Freeze parameters and use evaluation mode. No optimizer or weight updates.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = ResnetGenerator(3, 3, ngf=64, norm_layer=nn.InstanceNorm2d, n_blocks=9)
state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
expected = generator.state_dict()
# Older checkpoints contain InstanceNorm statistics unused by this architecture.
obsolete = [key for key in state if key not in expected]
for key in obsolete:
    module_name, buffer_name = key.rsplit(".", 1)
    module = generator.get_submodule(module_name)
    if not isinstance(module, nn.InstanceNorm2d) or buffer_name not in (
        "running_mean", "running_var", "num_batches_tracked"
    ):
        raise ValueError(f"Unexpected checkpoint entry: {key}")
    del state[key]
generator.load_state_dict(state, strict=True)
generator.requires_grad_(False)
generator.eval()
generator.to(device)
print(f"Loaded {len(state)} parameter tensors; omitted {len(obsolete)} legacy buffers.")
print(f"Checkpoint SHA256: {hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()}")
print(f"Device: {device}; PyTorch: {torch.__version__}")


## Photo-to-painting conversion

Use the same RGB conversion, [-1, 1] normalization, inference, and uint8 conversion as notebook 02. No random cropping or flipping.


In [ ]:
def translate_photo(generator, image_path, device):
    """Translate one RGB photo to a 256×256 RGB painting without augmentation."""
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        if image.size != (256, 256):
            image = image.resize((256, 256), Image.Resampling.BICUBIC)
        pixels = np.asarray(image, dtype=np.float32) / 127.5 - 1.0
    tensor = torch.from_numpy(pixels.transpose(2, 0, 1).copy()).unsqueeze(0).to(device)
    with torch.inference_mode():
        result = generator(tensor)[0].cpu()
    if not torch.isfinite(result).all():
        raise ValueError(f"Nonfinite output for {image_path}")
    pixels = ((result.permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    return Image.fromarray(pixels)


## Generate the output ZIP

- Transform all photos once and write JPG bytes directly into the output ZIP.
- Temporary files stay outside `/kaggle/working`.
- On success, the completed file is copied into place as `images.zip`.


In [ ]:
start = time.perf_counter()
with ZipFile(temporary_output_zip_path, "w", compression=ZIP_STORED) as output_zip:
    for index, image_path in enumerate(photo_paths, 1):
        painting = translate_photo(generator, image_path, device)
        if painting.size != (256, 256) or painting.mode != "RGB":
            raise ValueError(f"Invalid generated image: {image_path.name}")
        buffer = BytesIO()
        painting.save(buffer, format="JPEG", quality=95)
        output_zip.writestr(image_path.name, buffer.getvalue())
        if index % 500 == 0 or index == len(photo_paths):
            print(f"Generated {index:,}/{len(photo_paths):,} in {time.perf_counter() - start:.1f}s", flush=True)


## Validate the output ZIP


In [ ]:
with ZipFile(temporary_output_zip_path) as output_zip:
    names = output_zip.namelist()
    assert len(names) == len(set(names)) == len(photo_paths)
    assert set(names) == {path.name for path in photo_paths}
    assert output_zip.testzip() is None
    for name in names:
        with Image.open(BytesIO(output_zip.read(name))) as image:
            assert image.format == "JPEG" and image.mode == "RGB" and image.size == (256, 256), name
            image.load()
print(f"Validated {len(names):,} RGB JPEGs, all 256×256.")


In [ ]:
# Copy across filesystems if Kaggle scratch and working storage differ.
from shutil import copyfile

copyfile(temporary_output_zip_path, output_zip_path)
working_files = sorted(path.relative_to(output_dir).as_posix()
                       for path in output_dir.rglob("*") if path.is_file())
if working_files != ["images.zip"]:
    raise ValueError(f"Expected only images.zip in working output; found {working_files}. Remove unrelated output files before saving the final run.")
print(f"Output ZIP: {output_zip_path} ({output_zip_path.stat().st_size / 1024**2:.1f} MiB)")


## Save and submit on Kaggle

- Run all cells and confirm the final validation succeeds.
- Choose **Save Version → Save & Run All**.
- After the saved run finishes, confirm its output contains only `images.zip`.
- Use **Submit to Competition** / **Make a Submission** for that saved output, then check the competition score.
- These instructions do not submit automatically. No score is claimed here.


## Attribution and license

Original code: Jun-Yan Zhu and Taesung Park, [pytorch-CycleGAN-and-pix2pix](https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix).

<details>
<summary>Original license</summary>

```text
Copyright (c) 2017, Jun-Yan Zhu and Taesung Park
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.


--------------------------- LICENSE FOR pix2pix --------------------------------
BSD License

For pix2pix software
Copyright (c) 2016, Phillip Isola and Jun-Yan Zhu
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

----------------------------- LICENSE FOR DCGAN --------------------------------
BSD License

For dcgan.torch software

Copyright (c) 2015, Facebook, Inc. All rights reserved.

Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:

Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer.

Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution.

Neither the name Facebook nor the names of its contributors may be used to endorse or promote products derived from this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

```

</details>
